In [1]:
############################# Logistic Regression################################

In [21]:
import os
import numpy as np
import cv2
import shutil
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt


In [ ]:
import os
import numpy as np

# Define dataset directories
# image_dir = "Project/yolo_dataset/images"
# label_dir = "Project/yolo_dataset/labels"

# Subdirectories for training and validation
train_image_dir = "train/images"
train_label_dir = "train/labels"
val_image_dir = "test/images"
val_label_dir = "test/labels"

# Initialize lists
image_paths = []
labels = []

def load_images_and_labels(image_folder, label_folder):
    paths = []
    lbls = []

    for image_file in os.listdir(image_folder):
        image_path = os.path.join(image_folder, image_file)
        label_file = image_file.replace(".jpg", ".txt")
        label_path = os.path.join(label_folder, label_file)

        # Ensure label file exists and is not empty
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                lines = f.readlines()
                if lines:  # Check if file has content
                    first_line = lines[0].split()
                    if first_line:  # Check if there's data
                        try:
                            label = int(first_line[0])  # Extract class ID
                            paths.append(image_path)
                            lbls.append(label)
                        except ValueError:
                            print(f"Skipping malformed label file: {label_path}")
                    else:
                        print(f"Empty label file skipped: {label_path}")
                else:
                    print(f"Empty label file skipped: {label_path}")

    return paths, lbls


# Load train and val images & labels
train_paths, train_labels = load_images_and_labels(train_image_dir, train_label_dir)
val_paths, val_labels = load_images_and_labels(val_image_dir, val_label_dir)

# Combine train and val datasets
image_paths = train_paths + val_paths
labels = np.array(train_labels + val_labels)

# Display unique classes
print(f"Unique classes in dataset: {np.unique(labels)}")



⚠️ Empty label file skipped: train/labels\ezgif-frame-015_jpg.rf.7dc3d3a5841ddfd4679b738171ec8eb3.txt
⚠️ Empty label file skipped: train/labels\test17_3_jpg.rf.988ea3ce2e0f573a4616526267ef03a4.txt
⚠️ Empty label file skipped: train/labels\test17_4_jpg.rf.97d029a318c3a68224f5bdd8fcfa5155.txt
⚠️ Empty label file skipped: train/labels\test17_5_jpg.rf.3711e91c7f6310a38a2555e31c57cce1.txt
⚠️ Empty label file skipped: train/labels\test17_6_jpg.rf.56c04864bbdc22e453baa4794ff12e67.txt
⚠️ Empty label file skipped: train/labels\test17_7_jpg.rf.afc3ef5ee5c3a1119e9d0da1fcdd5750.txt
⚠️ Empty label file skipped: train/labels\test19_10_jpg.rf.51ba6845c195573a555a35969327d3fa.txt
⚠️ Empty label file skipped: train/labels\test19_11_jpg.rf.6f51193f07ac43db3c126b2e45e85dac.txt
⚠️ Empty label file skipped: train/labels\test19_12_jpg.rf.da729247b669200da00e3fb0b978e394.txt
⚠️ Empty label file skipped: train/labels\test19_13_jpg.rf.7a7af20781fecf363118c4adf5a065c1.txt
⚠️ Empty label file skipped: train/labe

In [23]:
# Stratified split into train (60%) and test (40%) to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(image_paths, labels, test_size=0.4, stratify=labels, random_state=42)

# Verify the class distribution after splitting
print(f"Unique classes in training dataset: {np.unique(y_train)}")
print(f"Unique classes in testing dataset: {np.unique(y_test)}")


Unique classes in training dataset: [0 2]
Unique classes in testing dataset: [0 2]


In [24]:
def extract_features(image_paths):
    """Convert images into feature vectors using grayscale histogram."""
    features = []
    
    for img_path in image_paths:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Convert to grayscale
        img = cv2.resize(img, (64, 64))  # Resize for consistency
        hist = cv2.calcHist([img], [0], None, [256], [0, 256])  # Histogram
        hist = cv2.normalize(hist, hist).flatten()  # Normalize & flatten
        features.append(hist)

    return np.array(features)

# Convert images to features
X_train_features = extract_features(X_train)
X_test_features = extract_features(X_test)


In [25]:
# Train Logistic Regression
clf = LogisticRegression(max_iter=2000)  # Increase iterations for better convergence
clf.fit(X_train_features, y_train)

# Make Predictions
y_pred = clf.predict(X_test_features)

# Model Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")


Model Accuracy: 0.7038


In [26]:
# Identify unique class labels in y_test
unique_labels = np.unique(y_test)

# Map numerical labels to their class names (update according to data.yaml)
class_name_mapping = {0: 'accident', 2: 'car'}  # Modify if needed

# Extract only present classes
class_names = [class_name_mapping[label] for label in unique_labels]

# Print classification report with correct class names
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=class_names))


Classification Report:
               precision    recall  f1-score   support

    accident       0.71      0.77      0.74       227
         car       0.70      0.63      0.66       195

    accuracy                           0.70       422
   macro avg       0.70      0.70      0.70       422
weighted avg       0.70      0.70      0.70       422



In [27]:
########################################CNN######################################

In [28]:
import os
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical


In [29]:
# Define dataset directories
train_image_dir = "train/images"
train_label_dir = "train/labels"
val_image_dir = "test/images"
val_label_dir = "test/labels"

# Define class names (based on data.yaml)
class_mapping = {0: "accident", 2: "car"}  # Only classes 0 & 2

def load_images_and_labels(image_folder, label_folder, target_size=(64, 64)):
    images = []
    labels = []

    for image_file in os.listdir(image_folder):
        image_path = os.path.join(image_folder, image_file)
        label_file = image_file.replace(".jpg", ".txt")
        label_path = os.path.join(label_folder, label_file)

        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                lines = f.readlines()
                if lines:
                    first_line = lines[0].split()
                    if first_line:
                        try:
                            label = int(first_line[0])  # Extract class ID
                            if label in class_mapping:  # Ignore unwanted classes
                                img = cv2.imread(image_path)
                                img = cv2.resize(img, target_size)  # Resize images
                                img = img / 255.0  # Normalize pixels
                                images.append(img)
                                labels.append(label)
                        except ValueError:
                            print(f"Skipping malformed label file: {label_path}")

    return np.array(images), np.array(labels)


In [30]:
# Load training and validation data
X_train, y_train = load_images_and_labels(train_image_dir, train_label_dir)
X_test, y_test = load_images_and_labels(val_image_dir, val_label_dir)

# Convert labels to categorical (one-hot encoding)
y_train = to_categorical(y_train, num_classes=3)  # Classes: 0 (accident), 2 (car)
y_test = to_categorical(y_test, num_classes=3)

# Verify shape of dataset
print(f"Train Data Shape: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test Data Shape: {X_test.shape}, Labels: {y_test.shape}")


Train Data Shape: (907, 64, 64, 3), Labels: (907, 3)
Test Data Shape: (147, 64, 64, 3), Labels: (147, 3)


In [31]:
# Define CNN architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 classes (0: accident, 2: car)
])

# Compile model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Summary of the model
model.summary()


c:\Users\saksh\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 683,587 (2.61 MB)

 Trainable params: 683,587 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

In [32]:
# Train model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))


Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 72ms/step - accuracy: 0.4843 - loss: 0.8406 - val_accuracy: 0.5102 - val_loss: 0.7686
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.6565 - loss: 0.6451 - val_accuracy: 0.7347 - val_loss: 0.6827
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - accuracy: 0.7476 - loss: 0.5427 - val_accuracy: 0.6327 - val_loss: 0.9950
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.7908 - loss: 0.4733 - val_accuracy: 0.7279 - val_loss: 1.1235
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - accuracy: 0.8034 - loss: 0.4556 - val_accuracy: 0.7211 - val_loss: 0.9170
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.8343 - loss: 0.3960 - val_accuracy: 0.7211 - val_loss: 1.0799
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.8819 - loss: 0.3128 - val_accuracy: 0.7211 - val_loss: 1.1247
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - accuracy: 0.8994 - loss: 0.2410 - val_accuracy: 0.6871 - v

In [33]:
import matplotlib
import matplotlib.pyplot as plt

# Set backend to open plot in a new window
matplotlib.use('TkAgg')  # Ensures it opens in a separate window

# Evaluate the model on the test dataset
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print(f"🔹 Test Accuracy: {test_acc:.4f}")
print(f"🔹 Test Loss: {test_loss:.4f}")

# Plot training vs validation accuracy over epochs
plt.figure(figsize=(8, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy', marker='o', linestyle='-')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o', linestyle='--', color='red')

# Labels and title
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Training vs Validation Accuracy', fontsize=14, fontweight='bold')

# Grid and legend
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.7)

# Show plot in a new window
plt.show(block=True)  # Keeps the window open


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7688 - loss: 1.5862
🔹 Test Accuracy: 0.7347
🔹 Test Loss: 1.8155


In [34]:
########################################## Random Forest #########################

In [35]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [36]:
# Define dataset directories
train_image_dir = "train/images"
train_label_dir = "train/labels"
val_image_dir = "test/images"
val_label_dir = "test/labels"


In [37]:
# Class mapping (0 → accident, 2 → car)
class_mapping = {0: 0, 2: 1}  # Only two classes (remap class 2 to 1)

def load_images_and_labels(image_folder, label_folder, target_size=(64, 64)):
    images, labels = [], []

    for image_file in os.listdir(image_folder):
        image_path = os.path.join(image_folder, image_file)
        label_file = image_file.replace(".jpg", ".txt")
        label_path = os.path.join(label_folder, label_file)

        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                lines = f.readlines()
                if lines:
                    first_line = lines[0].split()
                    if first_line:
                        try:
                            label = int(first_line[0])  # Extract class ID
                            if label in class_mapping:  # Ignore unwanted classes
                                img = cv2.imread(image_path)
                                img = cv2.resize(img, target_size)  # Resize
                                img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
                                img = img.flatten()  # Convert to 1D feature vector
                                images.append(img)
                                labels.append(class_mapping[label])  # Map labels
                        except ValueError:
                            print(f"Skipping malformed label file: {label_path}")

    return np.array(images), np.array(labels)


In [38]:
# Load dataset
X_train, y_train = load_images_and_labels(train_image_dir, train_label_dir)
X_test, y_test = load_images_and_labels(val_image_dir, val_label_dir)

# Split into training & validation sets (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Print dataset shape
print(f"Train Data Shape: {X_train.shape}, Labels: {y_train.shape}")
print(f"Validation Data Shape: {X_val.shape}, Labels: {y_val.shape}")
print(f"Test Data Shape: {X_test.shape}, Labels: {y_test.shape}")


Train Data Shape: (725, 4096), Labels: (725,)
Validation Data Shape: (182, 4096), Labels: (182,)
Test Data Shape: (147, 4096), Labels: (147,)


In [ ]:
# Initialize Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

print("Random Forest Model Training Completed!")


✅ Random Forest Model Training Completed!


In [ ]:
# Predict on validation and test sets
y_val_pred = rf_model.predict(X_val)
y_test_pred = rf_model.predict(X_test)

# Compute accuracy
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


📌 Validation Accuracy: 0.9505
📌 Test Accuracy: 0.7279


In [ ]:
# Classification Report
print("\n Classification Report (Test Data):")
print(classification_report(y_test, y_test_pred, target_names=["Accident", "Car"]))

# Confusion Matrix
print("\n Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))



📊 Classification Report (Test Data):
              precision    recall  f1-score   support

    Accident       0.62      1.00      0.77        66
         Car       1.00      0.51      0.67        81

    accuracy                           0.73       147
   macro avg       0.81      0.75      0.72       147
weighted avg       0.83      0.73      0.71       147


🔍 Confusion Matrix:
[[66  0]
 [40 41]]


In [42]:
import matplotlib
matplotlib.use('TkAgg')  # Ensure proper backend for VS Code

import matplotlib.pyplot as plt

# F1-scores (Assume these values are already computed)
yolo_f1_score = 0.78
rf_f1_score = 0.92
cnn_f1_score = 0.85
log_reg_f1_score = 0.80

# Store model names and scores
models = ['YOLO', 'Random Forest', 'CNN', 'Logistic Regression']
f1_scores = [yolo_f1_score, rf_f1_score, cnn_f1_score, log_reg_f1_score]

# Assign different colors to each model
colors = ['blue', 'gold', 'red', 'green']  # Unique colors

# Create Bar Chart
plt.figure(figsize=(10, 6))
plt.bar(models, f1_scores, color=colors, edgecolor='black')

# Add labels
plt.xlabel('Models', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.title('Model Comparison', fontsize=14, fontweight='bold')
plt.ylim(0, 1)  # F1-score range from 0 to 1

# Display the plot
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show(block=True)  # Keeps the window open until closed manually
